# Assignment 6 — Advanced Smoothing Techniques

Use the **four n-gram models** from the previous assignment (unigram, bigram, trigram, quadrigram) and test them on the **test** and **development/validation** sets with:

1. Interpolated smoothing (bigram, trigram, quadrigram)
2. Good-Turing smoothing
3. Katz backoff smoothing
4. Stupid backoff smoothing
5. Kneser-Ney smoothing

Data: 1,000,000 tokenized Telugu sentences from Assignment 1, split like Assignment 4/5 (1,000 test, 1,000 development, rest training).


## 0. Imports

In [1]:
import os
import math
import time
import random
from collections import Counter, defaultdict

import pandas as pd

MIN_P = 1e-12
D_KN = 0.75
STUPID_LAMBDA = 0.4

print("Setup complete.")

Setup complete.


## 1. Load data, split, and build the 4 n-gram models

In [2]:
data_path = os.path.join(
    "..", "Assignment1", "output", "indiccorp", "indiccorp_te_tokenized.parquet"
)
df = pd.read_parquet(data_path, columns=["tokenized_sentence"]).head(1_000_000)

all_sents = []
for text in df["tokenized_sentence"]:
    if isinstance(text, str):
        tokens = text.split()
        if tokens:
            all_sents.append(tokens)

print("Sentences loaded:", len(all_sents))
print("Example:", all_sents[0])

random.seed(42)
random.shuffle(all_sents)
test_set = all_sents[:1000]
dev_set = all_sents[1000:2000]
train_set = all_sents[2000:]
del all_sents, df

print("Train / dev / test:", len(train_set), len(dev_set), len(test_set))

Sentences loaded: 1000000
Example: ['అమెరికా', 'అధ్యక్షుడు', 'డొనాల్డ్', 'ట్రంప్', 'కు', 'రాష్ట్రపతి', 'భవన్', 'వద్ద', 'ఘనస్వాగతం', 'లభించింది', '.']


Train / dev / test: 998000 1000 1000


In [3]:
vocab = set()
for s in train_set:
    vocab.update(s)
vocab.update(["<UNK>", "<s>", "</s>"])
V = len(vocab)
print("Vocabulary size V:", V)


def replace_unk(sentence):
    return [w if w in vocab else "<UNK>" for w in sentence]


def pad(sentence, n):
    return ["<s>"] * (n - 1) + sentence + ["</s>"]


# models[n] = (ngram_counts, context_counts)
models = {n: (Counter(), Counter()) for n in [1, 2, 3, 4]}

t0 = time.time()
for s in train_set:
    s2 = replace_unk(s)
    for n in [1, 2, 3, 4]:
        ngc, cc = models[n]
        padded = pad(s2, n)
        for i in range(n - 1, len(padded)):
            ng = tuple(padded[i - n + 1 : i + 1])
            ngc[ng] += 1
            cc[ng[:-1]] += 1

N_TOKENS = sum(models[1][0].values())
print("Training tokens N:", N_TOKENS)
for n, name in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram"), (4, "Quadrigram")]:
    print(f"{name}: {len(models[n][0]):,} n-grams")
print(f"Counts built in {time.time() - t0:.1f}s")

Vocabulary size V: 591166


Training tokens N: 13330633
Unigram: 591,164 n-grams
Bigram: 4,777,970 n-grams
Trigram: 8,394,190 n-grams
Quadrigram: 10,141,000 n-grams
Counts built in 164.4s


In [4]:
def perplexity(sentences, n, prob_fn):
    log_sum, count = 0.0, 0
    for s in sentences:
        padded = pad(replace_unk(s), n)
        for i in range(n - 1, len(padded)):
            ng = tuple(padded[i - n + 1 : i + 1])
            p = prob_fn(ng, n)
            if p < MIN_P:
                p = MIN_P
            log_sum += math.log(p)
            count += 1
    return math.exp(-log_sum / count)


def mle(ng, n):
    ngc, cc = models[n]
    c = ngc[ng]
    if n == 1:
        return c / N_TOKENS if N_TOKENS else 0.0
    ctx_c = cc[ng[:-1]]
    return c / ctx_c if ctx_c else 0.0

## 2. Interpolated smoothing (bigram, trigram, quadrigram)

Linear interpolation of MLE estimates from order \(n\) down to unigrams, with equal weights \(\lambda = 1/n\):

$$P_{\text{interp}}(w_n \mid w_1^{n-1}) = \sum_{k=1}^{n} \frac{1}{n}\, P_{\text{MLE}}^{(k)}(w_n \mid w_{n-k+1}^{n-1})$$


In [5]:
def interpolated_prob(ng, n):
    p = 0.0
    lam = 1.0 / n
    for order in range(1, n + 1):
        p += lam * mle(ng[-order:], order)
    return p if p > 0 else (1.0 / V)


print("Interpolated smoothing")
print(f"{'Model':<12}{'Dev PP':>14}{'Test PP':>14}")
print("-" * 40)
interp_results = {}
for n, name in [(2, "Bigram"), (3, "Trigram"), (4, "Quadrigram")]:
    t0 = time.time()
    pp_dev = perplexity(dev_set, n, interpolated_prob)
    pp_test = perplexity(test_set, n, interpolated_prob)
    interp_results[name] = (pp_dev, pp_test)
    print(f"{name:<12}{pp_dev:>14.2f}{pp_test:>14.2f}   ({time.time()-t0:.1f}s)")

Interpolated smoothing
Model               Dev PP       Test PP
----------------------------------------
Bigram              784.01        797.76   (0.2s)


Trigram             632.01        673.71   (0.2s)


Quadrigram          704.02        757.54   (0.2s)


## 3. Good-Turing smoothing

Replace count \(c\) by \(c^* = (c+1) N_{c+1}/N_c\). Unseen n-grams share leftover probability mass uniformly over the rest of the vocabulary.


In [6]:
# Nr[n][c] = number of n-grams that occurred c times
Nr = {}
for n in [1, 2, 3, 4]:
    Nr[n] = Counter(models[n][0].values())


def c_star(c, n):
    if c <= 0:
        return 0.0
    n_c = Nr[n].get(c, 0)
    n_c1 = Nr[n].get(c + 1, 0)
    if n_c > 0 and n_c1 > 0 and c <= 5:
        return (c + 1) * n_c1 / n_c
    return float(c)


# leftover mass per context: 1 - sum_w c*(ctx,w) / C(ctx)
gt_follow = {n: Counter() for n in [1, 2, 3, 4]}
gt_disc_sum = {n: Counter() for n in [1, 2, 3, 4]}
for n in [1, 2, 3, 4]:
    ngc, cc = models[n]
    for ng, c in ngc.items():
        ctx = ng[:-1]
        gt_follow[n][ctx] += 1
        gt_disc_sum[n][ctx] += c_star(c, n)


def gt_prob(ng, n):
    ngc, cc = models[n]
    c = ngc[ng]
    ctx = ng[:-1]
    ctx_c = N_TOKENS if n == 1 else cc[ctx]
    if ctx_c == 0:
        return 1.0 / V
    if c > 0:
        return c_star(c, n) / ctx_c
    leftover = 1.0 - (gt_disc_sum[n][ctx] / ctx_c)
    unseen = V - gt_follow[n][ctx]
    if leftover <= 0 or unseen <= 0:
        return 1.0 / V
    return leftover / unseen


print("Good-Turing smoothing")
print(f"{'Model':<12}{'Dev PP':>14}{'Test PP':>14}")
print("-" * 40)
gt_results = {}
for n, name in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram"), (4, "Quadrigram")]:
    t0 = time.time()
    pp_dev = perplexity(dev_set, n, gt_prob)
    pp_test = perplexity(test_set, n, gt_prob)
    gt_results[name] = (pp_dev, pp_test)
    print(f"{name:<12}{pp_dev:>14.2f}{pp_test:>14.2f}   ({time.time()-t0:.1f}s)")

Good-Turing smoothing
Model               Dev PP       Test PP
----------------------------------------
Unigram            3090.18       2964.93   (0.1s)
Bigram             1313.46       1379.93   (0.1s)


Trigram            8348.10       9081.00   (0.1s)
Quadrigram        38981.25      42909.96   (0.1s)


## 4. Katz backoff smoothing

Good-Turing discount if the n-gram was seen; otherwise back off to the (n-1)-gram, scaled by leftover mass \(lpha(h)\).


In [7]:
katz_alpha = {n: {} for n in [2, 3, 4]}

for n in [2, 3, 4]:
    ngc, cc = models[n]
    seen_disc = defaultdict(float)
    for ng, c in ngc.items():
        ctx = ng[:-1]
        ctx_c = cc[ctx]
        seen_disc[ctx] += (c_star(c, n) / ctx_c) if ctx_c else 0.0
    for ctx, ctx_c in cc.items():
        beta = max(0.0, 1.0 - seen_disc[ctx])
        katz_alpha[n][ctx] = beta


def katz_prob(ng, n):
    ngc, cc = models[n]
    c = ngc[ng]
    if n == 1:
        return gt_prob(ng, 1)
    ctx = ng[:-1]
    ctx_c = cc[ctx]
    if c > 0 and ctx_c > 0:
        return c_star(c, n) / ctx_c
    lower = katz_prob(ng[1:], n - 1)
    return katz_alpha[n].get(ctx, 1.0) * lower


print("Katz backoff smoothing")
print(f"{'Model':<12}{'Dev PP':>14}{'Test PP':>14}")
print("-" * 40)
katz_results = {}
for n, name in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram"), (4, "Quadrigram")]:
    t0 = time.time()
    pp_dev = perplexity(dev_set, n, katz_prob)
    pp_test = perplexity(test_set, n, katz_prob)
    katz_results[name] = (pp_dev, pp_test)
    print(f"{name:<12}{pp_dev:>14.2f}{pp_test:>14.2f}   ({time.time()-t0:.1f}s)")

Katz backoff smoothing
Model               Dev PP       Test PP
----------------------------------------
Unigram            3090.18       2964.93   (0.0s)
Bigram              550.79        542.20   (0.1s)
Trigram             467.51        476.13   (0.1s)


Quadrigram          510.82        519.50   (0.1s)


## 5. Stupid backoff smoothing

If the n-gram count is positive, use MLE; otherwise multiply by \(0.4\) and back off (Brants et al.). This is a **score**, not a normalized probability.


In [8]:
def stupid_prob(ng, n):
    ngc, cc = models[n]
    c = ngc[ng]
    if n == 1:
        return (c + 1) / (N_TOKENS + V)  # avoid zeros at the unigram floor
    ctx_c = cc[ng[:-1]]
    if c > 0 and ctx_c > 0:
        return c / ctx_c
    return STUPID_LAMBDA * stupid_prob(ng[1:], n - 1)


print("Stupid backoff smoothing")
print(f"{'Model':<12}{'Dev PP':>14}{'Test PP':>14}")
print("-" * 40)
stupid_results = {}
for n, name in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram"), (4, "Quadrigram")]:
    t0 = time.time()
    pp_dev = perplexity(dev_set, n, stupid_prob)
    pp_test = perplexity(test_set, n, stupid_prob)
    stupid_results[name] = (pp_dev, pp_test)
    print(f"{name:<12}{pp_dev:>14.2f}{pp_test:>14.2f}   ({time.time()-t0:.1f}s)")

Stupid backoff smoothing
Model               Dev PP       Test PP
----------------------------------------
Unigram            4211.87       4260.23   (0.0s)
Bigram              644.26        674.27   (0.0s)
Trigram             695.70        767.46   (0.1s)
Quadrigram         1252.93       1401.14   (0.1s)


## 6. Kneser-Ney smoothing

Interpolated KN with discount \(D = 0.75\). Highest order uses raw counts; lower orders use **continuation counts** (how many distinct left contexts a word/n-gram appeared in).


In [9]:
# n1plus[n][ctx] = number of distinct words following ctx
n1plus = {n: Counter() for n in [1, 2, 3, 4]}
# cont_ngc[n][(w2..wn)] = number of distinct w1 such that (w1..wn) was seen
cont_ngc = {n: Counter() for n in [2, 3, 4]}
cont_cc = {n: Counter() for n in [2, 3, 4]}
cont_n1plus = {n: Counter() for n in [2, 3, 4]}

for n in [1, 2, 3, 4]:
    ngc, _ = models[n]
    for ng in ngc:
        n1plus[n][ng[:-1]] += 1
        if n >= 2:
            suffix = ng[1:]
            cont_ngc[n][suffix] += 1
            cont_cc[n][suffix[:-1]] += 1
    if n >= 2:
        for suf in cont_ngc[n]:
            cont_n1plus[n][suf[:-1]] += 1

cont_unigram_total = sum(cont_ngc[2].values())


def kn_prob(ng, n, use_cont=False):
    w_ctx = ng[:-1]
    if n == 1:
        # continuation unigram from bigrams
        c_cont = cont_ngc[2].get(ng, 0)
        return c_cont / cont_unigram_total if cont_unigram_total else (1.0 / V)

    if use_cont:
        c = cont_ngc[n + 1].get(ng, 0) if (n + 1) in cont_ngc else models[n][0][ng]
        ctx_c = cont_cc[n + 1].get(w_ctx, 0) if (n + 1) in cont_cc else models[n][1][w_ctx]
        uniq = cont_n1plus[n + 1].get(w_ctx, 0) if (n + 1) in cont_n1plus else n1plus[n][w_ctx]
    else:
        c = models[n][0][ng]
        ctx_c = models[n][1][w_ctx]
        uniq = n1plus[n][w_ctx]

    if ctx_c == 0:
        return kn_prob(ng[1:], n - 1, use_cont=True)

    p_lower = kn_prob(ng[1:], n - 1, use_cont=True)
    return max(c - D_KN, 0.0) / ctx_c + (D_KN * uniq / ctx_c) * p_lower


print("Kneser-Ney smoothing")
print(f"{'Model':<12}{'Dev PP':>14}{'Test PP':>14}")
print("-" * 40)
kn_results = {}
for n, name in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram"), (4, "Quadrigram")]:
    t0 = time.time()
    pp_dev = perplexity(dev_set, n, kn_prob)
    pp_test = perplexity(test_set, n, kn_prob)
    kn_results[name] = (pp_dev, pp_test)
    print(f"{name:<12}{pp_dev:>14.2f}{pp_test:>14.2f}   ({time.time()-t0:.1f}s)")

Kneser-Ney smoothing
Model               Dev PP       Test PP
----------------------------------------
Unigram            8698.13       9135.02   (0.1s)


Bigram              847.95        921.18   (0.2s)


Trigram             625.77        708.93   (0.3s)


Quadrigram          616.11        703.29   (0.5s)


## 7. Combined results

In [10]:
def row(method, name, pair):
    if pair is None:
        return f"{method:<18}{name:<12}{'—':>14}{'—':>14}"
    a, b = pair
    return f"{method:<18}{name:<12}{a:>14.2f}{b:>14.2f}"


print(f"{'Method':<18}{'Model':<12}{'Dev PP':>14}{'Test PP':>14}")
print("-" * 58)
for name in ["Bigram", "Trigram", "Quadrigram"]:
    print(row("Interpolated", name, interp_results.get(name)))
print()
for name in ["Unigram", "Bigram", "Trigram", "Quadrigram"]:
    print(row("Good-Turing", name, gt_results.get(name)))
print()
for name in ["Unigram", "Bigram", "Trigram", "Quadrigram"]:
    print(row("Katz backoff", name, katz_results.get(name)))
print()
for name in ["Unigram", "Bigram", "Trigram", "Quadrigram"]:
    print(row("Stupid backoff", name, stupid_results.get(name)))
print()
for name in ["Unigram", "Bigram", "Trigram", "Quadrigram"]:
    print(row("Kneser-Ney", name, kn_results.get(name)))

Method            Model               Dev PP       Test PP
----------------------------------------------------------
Interpolated      Bigram              784.01        797.76
Interpolated      Trigram             632.01        673.71
Interpolated      Quadrigram          704.02        757.54

Good-Turing       Unigram            3090.18       2964.93
Good-Turing       Bigram             1313.46       1379.93
Good-Turing       Trigram            8348.10       9081.00
Good-Turing       Quadrigram        38981.25      42909.96

Katz backoff      Unigram            3090.18       2964.93
Katz backoff      Bigram              550.79        542.20
Katz backoff      Trigram             467.51        476.13
Katz backoff      Quadrigram          510.82        519.50

Stupid backoff    Unigram            4211.87       4260.23
Stupid backoff    Bigram              644.26        674.27
Stupid backoff    Trigram             695.70        767.46
Stupid backoff    Quadrigram         1252.93       14

## 8. What the outputs mean

Lower **perplexity** means the model is less surprised by held-out text. Dev and test stay close for every method, so the ranking is not an accident of one split.

### Data
- 1,000,000 Telugu sentences; **998,000 train / 1,000 dev / 1,000 test**
- Vocabulary **V = 591,166**; **N = 13,330,633** training tokens
- Unique n-grams: unigram 0.59M, bigram 4.8M, trigram 8.4M, quadrigram 10.1M. That sparsity is why smoothing matters.

### 1. Interpolated smoothing
| Model | Dev PP | Test PP |
|---|---:|---:|
| Bigram | 784 | 798 |
| Trigram | **632** | **674** |
| Quadrigram | 704 | 758 |

Mixing MLE unigram + bigram + … means a zero higher-order count does not zero the whole probability. **Trigram interpolation is best** in this family. Quadrigram is a bit worse because equal weights still put 1/4 of the mass on very sparse 4-grams.

### 2. Good-Turing
| Model | Dev PP | Test PP |
|---|---:|---:|
| Unigram | 3090 | 2965 |
| Bigram | 1313 | 1380 |
| Trigram | 8348 | 9081 |
| Quadrigram | 38981 | 42910 |

Unigram GT is reasonable. **Higher-order GT gets worse** because this version does **not** back off: if a context is new, leftover mass is spread over almost all **V** words. That is the same kind of failure Laplace had in Assignment 4.

### 3. Katz backoff (best overall here)
| Model | Dev PP | Test PP |
|---|---:|---:|
| Unigram | 3090 | 2965 |
| Bigram | 551 | 542 |
| Trigram | **468** | **476** |
| Quadrigram | 511 | 520 |

Unigram Katz **equals** unigram GT (by design). For n>1, unseen n-grams **back off** to a shorter history instead of going uniform. That is why Katz trigram (~470) beats interpolation (~630) and GT trigram (~8500). Quadrigram is slightly worse than trigram: 4-word contexts are still rare, so the model spends more time on backoff.

### 4. Stupid backoff
| Model | Dev PP | Test PP |
|---|---:|---:|
| Unigram | 4212 | 4260 |
| Bigram | 644 | 674 |
| Trigram | 696 | 767 |
| Quadrigram | 1253 | 1401 |

Unigram matches **add-one** (the 1/V floor used to avoid zeros). Bigram is decent. Higher n gets worse because each miss multiplies by **0.4**, and the scores are **not** a true probability distribution, so this “perplexity” is only a rough score. It is simple and fast, not the most accurate.

### 5. Kneser-Ney
| Model | Dev PP | Test PP |
|---|---:|---:|
| Unigram | 8698 | 9135 |
| Bigram | 848 | 921 |
| Trigram | 626 | 709 |
| Quadrigram | **616** | **703** |

Standalone **unigram KN is supposed to look weak**: continuation probabilities are a backoff distribution, not a unigram LM. From bigram upward, KN **improves as n grows** (unlike GT), which is what continuation counts are for. It is close to interpolation and behind Katz on this corpus with a fixed discount D = 0.75.

### Takeaway
On this Telugu split, **Katz trigram** has the lowest perplexity (**dev 468 / test 476**). Interpolation and KN are the next useful methods. Good-Turing needs backoff (Katz) before it helps higher-order models. Stupid backoff is a cheap scoring heuristic, not a proper language model.
